# modular-diffusion: the addressed collective

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AbstractEyes/modular-diffusion/blob/v0.19.0/colab/addressed_collective.ipynb)

THE ADDRESSED COLLECTIVE (`geolip.modular_diffusion.experiments.addressed_collective`, built on `members` and `member_train`): nine small engines, one for each thing of the toy bed, and a base engine for the cells without a thing. Every engine receives the whole written layout of a picture as request tokens and finds the request addressed to it by address pairs inside attention; the collective serves every image token through the amplitude gate over the same addresses. A request either states its color or POINTS: it names another request of the picture by its address and is drawn in that request's color.

THE KINDS of engine (`members.VARIANTS`) and how this notebook trains and reads them:

| kind | told | sees | identity | pairs turned | noise | held pairs | reads after training |
|---|---|---|---|---|---|---|---|
| T1 | its own request | its cells | address and content | yes | stream | none | compose, exchange |
| T2 | every request | its cells | address | yes | device | plus4 | compose, address, exchange |
| T2K | every request | its cells | a name on one head | no | device | plus4 | compose, address |
| T3 | every request | every cell | address | yes | device | plus4 | compose, address, exchange |
| C1 | every request | its cells | address | no | device | plus4 | compose, address |
| C2 | every request | its cells | content | no | device | plus4 | compose, address |

Held pairs `plus4`: in training, class g never asks class (g + 4) mod 9; the compose job reads those pairs apart. Noise `stream` draws a step's noise on the CPU (the stored engines' order); `device` draws it on the card.

THE JOBS, each a folder under `OUT/runs` whose name says the kind, the bed, the seed and the steps:

| job | what it does |
|---|---|
| `stored_R1r_<bed>_s<seed>_<steps>` | the earlier toy collective's rotary-read collective (`colab/toy_collective.ipynb`, row R1r) read again on the same 500 lists by this bed's judges: 3,000-step engines of every selected bed and seed; for bed F seed 0 also the 1,500- and 6,000-step engines |
| `train_<kind>_<bed>_s<seed>_6000` | the ten engines of one kind trained as one stack (`variant=parts`), 6,000 steps, with snapshots of every engine at 1,500 and 3,000 steps; an interrupted training resumes from its stack checkpoint, on the machine that started it only |
| `compose_<kind>_<bed>_s<seed>_<steps>` | the collective of the training's engines at 6,000, 3,000 and 1,500 steps sampled on the 500 stored lists of 1, 2, 3, 4, 6 and 9 things and, for the kinds told every request, on lists that point (distinct colors; 3, 4, 6 and 9 things; with and without the held pairs), judged by the bed's judges, with the two address-blind references beside the pointed colors |
| `address_<kind>_<bed>_s<seed>_6000` | the address reads at the noise levels 0, 0.1 and 0.3: finding (the attention on the asked request), serving, both together with their controls, a missing and an unheld address, the tolerance control |
| `exchange_<kind>_<bed>_s0s1_6000` | one engine of the seed-0 collective exchanged for the seed-1 engine of its class: the change of every other member's contribution and the largest pixel difference outside its cells |

THE JUDGES and the stored engines are read from the public repository `SOURCE`; the results go to `REPO`.

THE SELECTION: four selectors at the top of the second cell (`BEDS`, `SEEDS`, `KINDS`, `READS`) choose a part of the run; nothing else changes. Several machines may each work a part at once: every finished job is shipped at once, a job finished on the Hub is fetched back and never run again, and an exchange job whose two trainings are neither on this machine nor on the Hub is listed as WAITING and skipped.

1. install at the release tag and print the environment;
2. the selection; the judges and stored engines fetched (the cell stops if one is missing); finished trainings fetched back; the job list; a timing smoke of a ten-engine stack; the projected minutes; the smoke shipped;
3. the job list, one job at a time; each finished job folder shipped at once; a progress line a minute; the summary; the machine is given back only when every finished job and the report were shipped.

THE MACHINE IS PART OF A TRAINING. With noise `device` a training's noise is drawn by the card's own generator, and the same seed can give other numbers on another card model. Every job therefore records the machine it ran on (`machine.json` in its folder: the card, its memory, the CPU count, the torch, CUDA and Python versions, the start and end times in UTC), the summary names the card of every training and says so when the trainings of one summary ran on different card models, and a training stays on one machine: its stack checkpoint is kept on this machine's own disk and is never shipped or fetched.

RESUME: run the cells again. Finished jobs keep their markers under `OUT` (keep `OUT` on this machine's own disk); on a new machine finished jobs are fetched back from the Hub, so nothing runs twice. A training interrupted on this machine continues from its stack checkpoint; on a new machine an unfinished training starts again from step 0.

In [ ]:
# C1 - install at the tag (extras on the name); torch is not a dependency, so the runtime's torch stays
%pip install -q "modular-diffusion[hub] @ git+https://github.com/AbstractEyes/modular-diffusion@v0.19.0"
import os, sys, re, time, json, shutil, threading
import torch, geolip.modular_diffusion as MD
assert tuple(int(x) for x in MD.__version__.split(".")) >= (0, 19, 0), "RESTART REQUIRED: Runtime > Restart session, then run C1 again"
import huggingface_hub
print("torch", torch.__version__, "| huggingface_hub", huggingface_hub.__version__, "| package", MD.__version__)
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print("card:", props.name, f"| memory {props.total_memory / 2**30:.1f} GiB | cpus: {os.cpu_count()}")
else:
    print(f"card: NONE (choose a GPU runtime) | cpus: {os.cpu_count()}")
OUT = "/content/addressed_collective"     # the output root; keep it the same to resume
os.makedirs(OUT, exist_ok=True)

THE SELECTION is the four selectors at the top (by default the first part: bed F, seed 0, the kinds T1, T2 and C1, every read). The judges of the selected beds and, for the stored reads, the stored engines are fetched from `SOURCE`; the cell stops if one is missing. Finished trainings are fetched back from `REPO` (also the other seed's, for the exchange jobs). The job list, in this order: the stored reads of the selected beds and seeds; then for each bed, seed and kind the training, its compose job at 6,000 steps, its address read, its compose jobs at 3,000 and 1,500 steps; last the exchange jobs whose two trainings are on this machine, on the Hub or in this list (the others are listed as WAITING). A read waits for its training (`needs=`). Then a smoke: the seconds a step of a ten-engine stack of kind T2 on this card, from which every training's minutes are projected; the reads are projected from measured seconds stated in the cell. The smoke's record is shipped (a check of the token before the long run).

In [ ]:
# C2 - the selection, the fetched files, the job list, the smoke, the projection
from huggingface_hub import snapshot_download
from geolip.modular_diffusion import hub, run
from geolip.modular_diffusion import collective as C
from geolip.modular_diffusion.experiments import toy_collective as T

# THE SELECTION: each selector chooses a part of the run; nothing else changes (several machines may take parts at once)
BEDS = ("F",)                                          # of "F" (flat), "C" (textured)
SEEDS = (0,)                                           # of 0, 1
KINDS = ("T1", "T2", "C1")                             # of T1, T2, T2K, T3, C1, C2
READS = ("stored", "compose", "address", "exchange")   # of the same four

RUNS, SMOKE, FETCHED = f"{OUT}/runs", f"{OUT}/smoke", f"{OUT}/source"
SOURCE = "AbstractPhil/geolip-multirotary-diffusion-collective"     # the judges and the stored engines (no token needed)
REPO = "AbstractPhil/geolip-multirotary-diffusion-collective-2"     # the results repository ("" = nothing shipped)
PREFIX = "modular_diffusion/conditioning/addressed_collective"
EARLIER = "modular_diffusion/conditioning/toy_collective"
SWAPPED = "modular_diffusion/conditioning/toy_swap"
NOT_SHIPPED = ("SHIPPED", "FROM_HUB", "stack_checkpoint*", "*.part")   # markers; the stack checkpoint (it stays on this machine)
DEVICE = "cuda"
EXTRA = ""                          # more arguments for every job of the addressed collective (empty in a real run)
EXTRA_STORED = ""                   # more arguments for the stored reads (the earlier experiment; empty in a real run)
STEPS, SNAPSHOTS = 6000, (1500, 3000)            # a training's steps and the steps of its snapshots
STORED_STEPS, DOSE_STEPS = 3000, (1500, 6000)    # the stored engines' steps; bed F seed 0's other lengths
LISTS = "count=500 chunk=128 sample_steps=50"    # the compose jobs' lists and sampling
ADDRESS = "k=256 ts=0.1,0.3"                     # the address jobs' rows a thing engine and noise levels
# MEASURED SECONDS, for the projection only
COMPOSE_S = 180.0      # a compose job on an L4 (measured); a stored read and an exchange job are projected at this too
STEP_S_L4 = 0.40       # a ten-engine training step on an L4 (expected); the smoke's measure on this card replaces it
ADDRESS_S = 190.0      # an address job on an L4 (not measured there: 364 s on a T4, scaled by the compose job's ratio of the two cards)
TRAIN_EXTRA_S = 120.0  # a training's own reads, snapshots and saves (an allowance)
# THE KINDS: (noise, held pairs, the reads after training)
KIND_SET = {"T1": ("stream", "", ("compose", "exchange")),
            "T2": ("device", "plus4", ("compose", "address", "exchange")),
            "T2K": ("device", "plus4", ("compose", "address")),
            "T3": ("device", "plus4", ("compose", "address", "exchange")),
            "C1": ("device", "plus4", ("compose", "address")),
            "C2": ("device", "plus4", ("compose", "address"))}
# THE JUDGES of each bed: (job, where it is in SOURCE); the first fills every key a single judge would
JUDGES = {"F": [("judge_F", EARLIER), ("judge_F_j1", SWAPPED), ("judge_F_j2", SWAPPED), ("judge_F_j3", SWAPPED),
                ("judge_F_j4", SWAPPED)],
          "C": [("judge_C", EARLIER)]}
assert set(BEDS) <= set(JUDGES) and set(SEEDS) <= {0, 1} and set(KINDS) <= set(KIND_SET), "a selector names an unknown value"
assert set(READS) <= {"stored", "compose", "address", "exchange"}, "READS names an unknown read"
assert len(set(BEDS)) == len(BEDS) and len(set(SEEDS)) == len(SEEDS) and len(set(KINDS)) == len(KINDS), "a selector names a value twice"
os.makedirs(RUNS, exist_ok=True)
mark = lambda t, m: os.path.exists(f"{RUNS}/{t}/{m}")
E = lambda t: f"{FETCHED}/{EARLIER}/{t}"

# the stored reads of the selection: (tag, the engines' steps, bed, seed, the things' job, the base's job)
STORED = []
if "stored" in READS:
    for b in BEDS:
        for s in SEEDS:
            STORED.append((f"stored_R1r_{b}_s{s}_{STORED_STEPS}", STORED_STEPS, b, s, f"atr_{b}_s{s}", f"abr_{b}_s{s}"))
            if b == "F" and s == 0:
                STORED += [(f"stored_R1r_F_s0_{d}", d, "F", 0, f"dose{d}_atr_F_s0", f"dose{d}_abr_F_s0") for d in DOSE_STEPS]
STORED_JOBS = [x for st in STORED for x in st[4:6]]
JUDGE_JOBS = [(t, where) for b in BEDS for t, where in JUDGES[b]]
def source_missing():
    # a judge is missing when its result or weights are not here; a stored job when its result or its engines are not
    miss = [f"the judge {t}" for t, where in JUDGE_JOBS
            if not (os.path.exists(f"{FETCHED}/{where}/{t}/result.json") and os.path.exists(f"{FETCHED}/{where}/{t}/judge.safetensors"))]
    for t in STORED_JOBS:
        found = T.index_engines(E(t))
        want = 9 if "_atr_" in f"_{t}" else 1
        if not os.path.exists(f"{E(t)}/result.json") or len(found) < want or not all(
                os.path.exists(f"{p}/{f}") for p in found.values() for f in C.ENGINE_FILES):
            miss.append(f"the stored engines of {t}")
    return sorted(set(miss))
if source_missing():
    snapshot_download(repo_id=SOURCE, allow_patterns=[f"{where}/{t}/*" for t, where in JUDGE_JOBS]
                      + [f"{EARLIER}/{t}/*" for t in STORED_JOBS], local_dir=FETCHED)
MISSING = source_missing()
assert not MISSING, f"the judges or stored engines are not all on the Hub ({', '.join(MISSING)}): STOP"
print("fetched: the judges", ", ".join(t for t, _ in JUDGE_JOBS) + ("; the stored engines " + ", ".join(STORED_JOBS) if STORED_JOBS else ""))
if REPO:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")          # placed in this process's environment only

def fetch_back(tags):
    # finished jobs come back from the Hub with their files, so nothing runs twice
    if not REPO:
        return
    want = [t for t in tags if not mark(t, "DONE")]
    if not want:
        return
    try:
        got = snapshot_download(repo_id=REPO, allow_patterns=[f"{PREFIX}/{t}/*" for t in want], local_dir=f"{OUT}/hub_back")
    except Exception as e:
        print("nothing fetched back from the Hub:", repr(e)[:200])
        return
    for t in want:
        src, dst = f"{got}/{PREFIX}/{t}", f"{RUNS}/{t}"
        if not os.path.exists(f"{src}/result.json"):
            continue
        shutil.copytree(src, dst, dirs_exist_ok=True)
        open(f"{dst}/DONE", "w").write("fetched back from the Hub\n")
        open(f"{dst}/FROM_HUB", "w").write("fetched back from the Hub\n")
        print(f"fetched back from the Hub: {t}")

def ship_job(tag, folder):
    if REPO and hub.ship(folder, REPO, f"{PREFIX}/{tag}", f"addressed collective: {tag}", ignore=NOT_SHIPPED):
        open(f"{folder}/SHIPPED", "w").write("shipped to the Hub\n")
        return True
    return False

# the job list: (tag, the program's arguments, the jobs it needs, what it is), in the order of the run
train_tag = lambda k, b, s: f"train_{k}_{b}_s{s}_{STEPS}"
judge_arg = lambda b: ",".join(f"{FETCHED}/{where}/{t}" for t, where in JUDGES[b])
fetch_back([train_tag(k, b, s) for b in BEDS for k in KINDS for s in (0, 1)])
JOBS, WAITING = [], []
def add(tag, args, needs, what):
    JOBS.append((tag, args, list(needs), what))
for tag, steps, b, s, th, ba in STORED:
    add(tag, f"toy_collective variant=compose row=R1r bed={b} seed={s} part_steps={steps} parts={E(th)},{E(ba)} "
             f"judge={judge_arg(b)} ns=1,2,3,4,6,9 {LISTS} device={DEVICE} det=on{EXTRA_STORED}", [], "stored")
for b in BEDS:
    for s in SEEDS:
        for k in KINDS:
            noise, held, reads = KIND_SET[k]
            common = f"kind={k} bed={b} seed={s} held={held} noise={noise} device={DEVICE} det=on"
            tt = train_tag(k, b, s)
            add(tt, f"addressed_collective variant=parts classes=0-8,base steps={STEPS} "
                    f"snapshots={','.join(map(str, SNAPSHOTS))} {common}{EXTRA}", [], "train")
            compose = lambda p: add(f"compose_{k}_{b}_s{s}_{p}", f"addressed_collective variant=compose part_steps={p} "
                                    f"parts={RUNS}/{tt} judge={judge_arg(b)} {LISTS} {common}{EXTRA}", [tt], "compose")
            if "compose" in READS and "compose" in reads:
                compose(STEPS)
            if "address" in READS and "address" in reads:
                add(f"address_{k}_{b}_s{s}_{STEPS}", f"addressed_collective variant=address part_steps={STEPS} "
                    f"parts={RUNS}/{tt} {ADDRESS} {common}{EXTRA}", [tt], "address")
            if "compose" in READS and "compose" in reads:
                compose(SNAPSHOTS[1])
                compose(SNAPSHOTS[0])
LISTED = {t for t, *_ in JOBS}
if "exchange" in READS:
    for b in BEDS:
        for k in KINDS:
            noise, held, reads = KIND_SET[k]
            if "exchange" not in reads:
                continue
            pair = [train_tag(k, b, s) for s in (0, 1)]
            tag = f"exchange_{k}_{b}_s0s1_{STEPS}"
            absent = [t for t in pair if t not in LISTED and not mark(t, "DONE")]
            if absent:
                WAITING.append((tag, absent))
                continue
            add(tag, f"addressed_collective variant=exchange kind={k} bed={b} seed=0 twin_seed=1 part_steps={STEPS} "
                     f"parts={RUNS}/{pair[0]},{RUNS}/{pair[1]} held={held} noise={noise} device={DEVICE} det=on{EXTRA}",
                [t for t in pair if t in LISTED], "exchange")
BY_TAG = {t: (a, n, w) for t, a, n, w in JOBS}
assert len(BY_TAG) == len(JOBS), "two jobs of one tag"
pos = {t: i for i, (t, *_) in enumerate(JOBS)}
assert all(pos[n] < pos[t] for t, _, ns, _ in JOBS for n in ns), "a job stands before a job it needs"
assert JOBS, "the selection holds no job: STOP"

# the smoke: the seconds a step of a ten-engine stack of kind T2 on this card (nothing kept but the result)
shutil.rmtree(SMOKE, ignore_errors=True)
smoke_args = (f"variant=timing kind=T2 stacks=10 steps=30 from_step=10 noise=device held=plus4 phases=0 bed={BEDS[0]} "
              f"device={DEVICE}{EXTRA}").split()
t0 = time.time()
sm = run.main("addressed_collective", smoke_args + [f"out={SMOKE}"])
STEP_S = float(sm["timing"][0]["seconds_a_step_stack"])
print(f"smoke: a ten-engine step of kind T2 takes {STEP_S:.4f} s on this card ({STEP_S_L4} s expected on an L4); "
      f"{time.time() - t0:.0f} s for the smoke")
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# the projection: a training from the smoke's seconds a step; the reads from the measured seconds above
SECONDS = {"train": STEPS * STEP_S + TRAIN_EXTRA_S, "compose": COMPOSE_S, "stored": COMPOSE_S, "exchange": COMPOSE_S,
           "address": COMPOSE_S if ADDRESS_S is None else ADDRESS_S}
COST = {t: SECONDS[w] for t, _, _, w in JOBS}
INSTRUMENTS = {"selection": {"beds": list(BEDS), "seeds": list(SEEDS), "kinds": list(KINDS), "reads": list(READS)},
               "jobs": [{"tag": t, "arguments": a, "needs": n, "what": w} for t, a, n, w in JOBS],
               "waiting": [{"tag": t, "needs": n} for t, n in WAITING], "smoke": sm["timing"][0],
               "seconds_a_step": STEP_S, "projected_seconds": COST,
               "measured_seconds": {"compose_l4": COMPOSE_S, "step_l4": STEP_S_L4, "address_l4": ADDRESS_S},
               "card": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
               "versions": {"torch": torch.__version__, "package": MD.__version__}}
json.dump(INSTRUMENTS, open(f"{SMOKE}/instruments.json", "w", encoding="utf-8"), indent=1, default=str)
print(f"the job list ({len(JOBS)} jobs; the minutes are projected on this card for the trainings, on an L4 for the reads):")
for i, (t, _, n, w) in enumerate(JOBS, 1):
    state = " DONE" + (" (fetched back)" if mark(t, "FROM_HUB") else "") if mark(t, "DONE") else ""
    print(f"  {i:2d}. {t}  ({COST[t] / 60:.0f} min{', after ' + ', '.join(n) if n else ''}){state}")
for t, n in WAITING:
    print(f"  WAITING (skipped, not an error): {t}, its trainings {', '.join(n)} are neither here nor on the Hub")
left = [t for t, *_ in JOBS if not mark(t, "DONE")]
by_what = {w: sum(COST[t] for t in left if BY_TAG[t][2] == w) / 60 for w in SECONDS}
print(f"projected for the {len(left)} jobs not done: about {sum(COST[t] for t in left) / 60:.0f} minutes with one runner ("
      + ", ".join(f"{w} {m:.0f}" for w, m in by_what.items() if m) + "; a training "
      f"{SECONDS['train'] / 60:.0f} min, a compose job {COMPOSE_S / 60:.0f} min"
      + (", an address job not measured: projected as a compose job" if ADDRESS_S is None else "") + ")")
if REPO:
    assert hub.ship(SMOKE, REPO, f"{PREFIX}/_smoke", "addressed collective: smoke run", ignore=NOT_SHIPPED), (
        "the smoke folder was not shipped: check the token and the repository before the long run")

One runner works the job list in order; a read starts only when its training is done. A job holding its finished marker is skipped; a training interrupted on this machine continues from its stack checkpoint when the cell runs again (the same arguments); on a new machine an unfinished training starts again from step 0, since its checkpoint never leaves the machine that started it. Before a job is shipped its folder gets `machine.json` (the card's name, its memory in GiB, the CPU count, the torch, CUDA and Python versions, the job's start and end times in UTC; on a machine without a card the card's name is null). Each finished job folder is shipped at once to `REPO` (the engines included, the stack checkpoint left out), with the token from the notebook's secret `HF_TOKEN` (placed in this process's environment only; the jobs themselves run without it). A progress line a minute names the running job, its step or phase, the minutes spent and the minutes left for it and for the list. At the end the summary (per training the card it ran on, said plainly when the trainings ran on different card models, and every engine's newborn and last own read and whether it descended; per compose job the things right pooled over the list sizes, whole scenes right at six requests, the stated and pointed colors right with both address-blind references and the pointed colors by the number of eligible requests; per address job and noise level the rows used, the four joint counts and the largest changes; per exchange job its differences), the roster, its log and the instruments are shipped to `_report`. The machine is given back (`RELEASE`) only when the report and every finished job were shipped; if an upload failed, the machine is kept and the cell names the jobs.

In [ ]:
# C3 - the job list, each finished job shipped, a progress line a minute, the summary, the machine
from geolip.modular_diffusion import jobs
RELEASE = True                     # give the machine back at the end (only when every finished job and the report were shipped)
PROGRESS_S = 60
MISSING = source_missing()
assert not MISSING, f"the judges or stored engines are not all here ({', '.join(MISSING)}): STOP before the roster"
tags = [t for t, *_ in JOBS]
roster = f"{OUT}/roster.txt"
cap = lambda t: max(30, int(3 * COST[t] / 60))
with open(roster, "w", encoding="utf-8", newline="\n") as fh:
    fh.write("\n".join(f"{t} | {cap(t)} | geolip.modular_diffusion.run | {a}" + (f" | needs={','.join(n)}" if n else "")
                       for t, a, n, _ in JOBS) + "\nEND\n")
fetch_back(tags)
for t in tags:      # a claim left by an interrupted session would hold its job forever
    if mark(t, "CLAIM") and not (mark(t, "DONE") or mark(t, "FAILED")):
        os.remove(f"{RUNS}/{t}/CLAIM")
from datetime import datetime, timezone
utc = lambda s: datetime.fromtimestamp(s, timezone.utc).isoformat(timespec="seconds")
def write_machine(tag, folder):
    # the machine a job ran on, written once (a job shipped again later keeps its first record)
    p = f"{folder}/machine.json"
    if os.path.exists(p):
        return
    cuda = torch.cuda.is_available()
    start = os.path.getmtime(f"{folder}/CLAIM") if os.path.exists(f"{folder}/CLAIM") else None
    rec = {"job": tag, "card": torch.cuda.get_device_name(0) if cuda else None,
           "card_memory_gib": round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2) if cuda else None,
           "cpus": os.cpu_count(), "torch": torch.__version__, "cuda": torch.version.cuda, "python": sys.version.split()[0],
           "package": MD.__version__, "start_utc": None if start is None else utc(start), "end_utc": utc(time.time())}
    json.dump(rec, open(p, "w", encoding="utf-8"), indent=1)
def after_job(tag, folder, ok):
    write_machine(tag, folder)
    if ok and REPO and not ship_job(tag, folder):
        print(f"{tag}: NOT SHIPPED (its folder stays on this machine: {folder})", flush=True)

shipped_or_fetched = lambda t: mark(t, "SHIPPED") or mark(t, "FROM_HUB")
for t in tags:      # a job that finished in an earlier session but whose upload failed: shipped again now
    if REPO and mark(t, "DONE") and not shipped_or_fetched(t):
        after_job(t, f"{RUNS}/{t}", True)
for t in tags:
    if mark(t, "DONE"):
        print(f"skipped, finished already ({'fetched back from the Hub' if mark(t, 'FROM_HUB') else 'on this machine'}): {t}")
    elif mark(t, "FAILED"):
        print(f"skipped, FAILED in an earlier session (remove its FAILED file to run it again): {t}")
for t, n in WAITING:
    print(f"WAITING, not run: {t} (its trainings {', '.join(n)} are neither here nor on the Hub)")

def where_is(t):
    # -> (its step or phase, minutes spent on it, minutes left for it), from its claim's time and its log's last line
    spent = (time.time() - os.path.getmtime(f"{RUNS}/{t}/CLAIM")) / 60 if mark(t, "CLAIM") else 0.0
    try:
        last = open(f"{RUNS}/{t}/log.txt", encoding="utf-8").read().strip().split("\n")[-1]
    except OSError:
        last = ""
    m = re.match(r"step ([\d,]+)/([\d,]+)", last)
    if m:
        n, total = int(m.group(1).replace(",", "")), int(m.group(2).replace(",", ""))
        s = re.search(r"left about (\d+) s", last)
        left = int(s.group(1)) / 60 if s else max(0.0, COST[t] * (total - n) / max(1, total) / 60)
        return f"step {n:,} of {total:,}", spent, left
    return (last.split(":")[0][:70] or "starting"), spent, max(0.0, COST[t] / 60 - spent)

t0 = time.time()
left0 = sum(COST[t] for t in tags if not (mark(t, "DONE") or mark(t, "FAILED")))
print(f"start: {sum(mark(t, 'DONE') for t in tags)}/{len(tags)} jobs already done; about {left0 / 60:.0f} min to run; one runner")
runner = threading.Thread(target=jobs.run_roster, daemon=True,
                          kwargs=dict(roster=roster, out=RUNS, stop=f"{OUT}/STOP", log=f"{OUT}/roster.log", card_room_mib=None,
                                      ram_free_gb=4.0, max_hours=24.0, claim=True, after_job=after_job, wait_s=5.0,
                                      env={"OMP_NUM_THREADS": "4", "MKL_NUM_THREADS": "4", "HF_HUB_OFFLINE": "1"}))
runner.start()
while runner.is_alive():
    runner.join(PROGRESS_S)
    if not runner.is_alive():
        break
    done, failed = sum(mark(t, "DONE") for t in tags), sum(mark(t, "FAILED") for t in tags)
    now = [t for t in tags if mark(t, "CLAIM") and not (mark(t, "DONE") or mark(t, "FAILED"))]
    rest = sum(COST[t] for t in tags if not (mark(t, "DONE") or mark(t, "FAILED")) and t not in now) / 60
    head = "between jobs"
    if now:
        phase, spent, left = where_is(now[0])
        head = f"{now[0]}: {phase}; {spent:.0f} min spent, about {left:.0f} min left for it"
        rest += left
    print(f"{head} | jobs done {done}/{len(tags)} (failed {failed}); {(time.time() - t0) / 60:.0f} min spent; about "
          f"{rest:.0f} min left for the list", flush=True)

# the summary
f4 = lambda v: "-" if v is None else (f"{v:.4f}" if isinstance(v, float) else str(v))
mean = lambda d: f4((d or {}).get("mean"))
def pooled(by_n):
    # the things right pooled over the list sizes: every right thing over every request
    right = sum(sum(v["per_list"]["n_right"]) for v in by_n.values())
    asked = sum(int(n) * len(v["per_list"]["n_right"]) for n, v in by_n.items())
    return right / asked if asked else None
def stored_line(by_n):
    return (f"things right {f4(pooled(by_n))} (pooled over n = {', '.join(by_n)}); whole scenes right at six requests "
            f"{mean(by_n.get('6', {}).get('scene_right'))}")
lines, CARDS = [], {}
for t in tags:
    try:            # a summary line never stops the report and the machine's return
        p = f"{RUNS}/{t}/result.json"
        if not os.path.exists(p):
            lines.append(f"{t}: NO RESULT" + (" (FAILED)" if mark(t, "FAILED") else ""))
            continue
        r = json.load(open(p, encoding="utf-8"))
        what = BY_TAG[t][2]
        if what == "train":
            es = r.get("engines") or []
            mp = f"{RUNS}/{t}/machine.json"
            card = json.load(open(mp, encoding="utf-8")).get("card") if os.path.exists(mp) else "unknown (no machine record)"
            CARDS[t] = card
            lines.append(f"{t}: on the card {card}; {len(es)} engines, {sum(bool(e['descended']) for e in es)} descended; own reads newborn -> last: "
                         + ", ".join(f"{'base' if e['cls'] is None else e['cls']} {e['newborn_read']:.5f} -> {e['last_read']:.5f}"
                                     f"{'' if e['descended'] else ' (NOT descended)'}" for e in es))
        elif what == "stored":
            lines.append(f"{t}: {stored_line(r['by_n'])}; inconclusive {r.get('inconclusive')}")
        elif what == "compose":
            lines.append(f"{t}: {stored_line(r['stored'])}; inconclusive {r.get('inconclusive')}")
            if not r["pointing"]:
                lines.append("    no lists that point (this kind is told its own request only)")
            for label, by_n in r["pointing"].items():
                for n, x in by_n.items():
                    by_k = ", ".join(f"k {k}: {mean(v['pointing_color_right'])} ({v['lists']} lists; 1/k {f4(v['one_over_k'])})"
                                     for k, v in (x.get("pointing_by_k") or {}).items())
                    lines.append(f"    {label.replace('_', ' ')} n={n}: stated colors right {mean(x['stated_color_right'])}; "
                                 f"pointed colors right {mean(x['pointing_color_right'])} against a guess among every other "
                                 f"stated color {f4(x['reference_all_stated']['mean'])} and a guess among the eligible "
                                 f"requests {f4(x['reference_eligible']['mean'])} ({x['reference_eligible']['lists']} lists); "
                                 f"by k: {by_k or '-'}")
        elif what == "address":
            find = r["find"]
            if "by_t" not in find:
                lines.append(f"{t}: {find.get('note')}")
            else:
                lines.append(f"{t}: the finding read (vary {find['vary']}; {find['k_rows_drawn_per_engine']} rows drawn an engine)")
            for tv, x in (find.get("by_t") or {}).items():
                bl = list(x["blocks"].values())
                lines.append(f"    t={tv}: rows used {x['rows']} of {x['rows_drawn']}; nearer the asked color under both asks "
                             f"{x['both_nearer']}, ask a only {x['only_a']}, ask b only {x['only_b']}, neither {x['neither']}; "
                             f"largest changes: raw answers {f4(x['max_abs_answer_delta'])}, attention weights "
                             f"{f4(max((b['max_abs_weight_delta'] for b in bl), default=None))}, pair-group scores "
                             f"{f4(max((b['max_abs_score_delta'] for b in bl), default=None))}; rows exactly equal by block "
                             f"{[b['rows_exactly_equal'] for b in bl]}")
        elif what == "exchange":
            c = r["chain"]
            one = {tv: f4(max(v["others"].values()) if v["others"] else None) for tv, v in r["one_step"].items()}
            lines.append(f"{t}: class {r['swap_class']} taken from seed {r['twin_seed']}: the largest difference outside its "
                         f"cells over the chain {f4(c['largest_outside'])} (before the clamp {f4(c['largest_outside_unclamped'])}); "
                         f"one step, the other members' largest change by t {one}")
    except Exception as e:
        lines.append(f"{t}: the summary line failed: {e!r}"[:300])
models = sorted({str(c) for c in CARDS.values()})
if len(models) > 1:
    lines.append("THE TRAININGS OF THIS SUMMARY RAN ON DIFFERENT CARD MODELS (" + "; ".join(
        f"{m}: {', '.join(t for t, c in CARDS.items() if str(c) == m)}" for m in models)
        + "): with noise device their numbers are not those of one card")
elif models:
    lines.append(f"every training of this summary ran on one card model: {models[0]}")
lines += [f"{t}: WAITING, not run (its trainings {', '.join(n)} are neither here nor on the Hub)" for t, n in WAITING]
summary = "\n".join(lines) + "\n"
print(summary)
REPORT = f"{OUT}/report"
os.makedirs(REPORT, exist_ok=True)
open(f"{REPORT}/summary.md", "w", encoding="utf-8").write(summary)
json.dump(INSTRUMENTS, open(f"{REPORT}/instruments.json", "w", encoding="utf-8"), indent=1, default=str)
for f in ("roster.txt", "roster.log"):
    if os.path.exists(f"{OUT}/{f}"):
        shutil.copy(f"{OUT}/{f}", f"{REPORT}/{f.replace('.log', '_log.txt')}")
for t in tags:      # the logs of the jobs that failed
    if mark(t, "FAILED"):
        for f in ("run.log", "log.txt"):
            if os.path.exists(f"{RUNS}/{t}/{f}"):
                shutil.copy(f"{RUNS}/{t}/{f}", f"{REPORT}/failed_{t}_{f.replace('.', '_')}.txt")
shipped = hub.ship(REPORT, REPO, f"{PREFIX}/_report", "addressed collective: summary, roster and instruments") if REPO else None
unshipped = [t for t in tags if mark(t, "DONE") and not shipped_or_fetched(t)]
unfinished = [t for t in tags if not mark(t, "DONE")]
print("report shipped:", shipped, "| finished jobs not on the Hub:", ", ".join(unshipped) or "none",
      "| jobs not finished:", ", ".join(unfinished) or "none")
if RELEASE and shipped and not unshipped and not unfinished:
    from google.colab import runtime
    runtime.unassign()
else:
    print("THE MACHINE IS KEPT:", "the report was not shipped" if not shipped else
          f"these finished jobs were not shipped: {', '.join(unshipped)} (run C3 again to retry)" if unshipped else
          f"these jobs did not finish: {', '.join(unfinished)}" if unfinished else "RELEASE is False")